# DEV · CardioRitmo: tamizaje de latidos (ECG5000)
**Responsable:** Francisco

**Especificaciones del cliente (prueba, umbral 0.5):** sensibilidad ≥ 0.95 · especificidad ≥ 0.95 · ≤ 20,000 parámetros entrenables.

In [ ]:
import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Datos (carga y división exactas del enunciado)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

ecg = pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv",
                  header=None)
X = ecg.iloc[:, :-1].to_numpy("float32")       # 140 puntos por latido
y = 1 - ecg.iloc[:, -1].to_numpy("int64")      # el archivo trae 1 = normal; aquí 1 = ANORMAL

# Prueba del cliente: 20 % estratificado. No la toquen hasta el final.
X_resto, X_prueba, y_resto, y_prueba = train_test_split(X, y, test_size=0.2, stratify=y,
                                                        random_state=42)
# Con X_resto y y_resto armen su entrenamiento y su validación como quieran.

## 2. Exploración
Tamaño, balance de clases, ejemplos y al menos 2 gráficas.

In [ ]:
import matplotlib.pyplot as plt
print("Forma:", X.shape)
print("Balance (1 = anormal):", pd.Series(y).value_counts(normalize=True).round(3).to_dict())
# TODO: barras de clases, latidos de ejemplo por clase, promedio ± desviación estándar por clase

## 3. Preparación
Validación estratificada desde `X_resto`. Los datos ya vienen normalizados.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_resto, y_resto, test_size=0.2,
                                            stratify=y_resto, random_state=42)
X_tr_s, X_val_s, X_prueba_s = (a[..., None] for a in (X_tr, X_val, X_prueba))  # (n, 140, 1) para Conv1D/RNN
print(X_tr.shape, X_val.shape, X_prueba.shape)

## 4. Métricas del cliente

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

def metricas(y_true, prob, umbral=0.5):
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens, esp = tp / (tp + fn), tn / (tn + fp)
    auc = roc_auc_score(y_true, prob) if len(set(prob)) > 1 else 0.5
    return {"sensibilidad": sens, "especificidad": esp, "auc": auc,
            "cumple_metricas": sens >= 0.95 and esp >= 0.95}

## 5. Línea base

In [ ]:
from sklearn.dummy import DummyClassifier
base = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(metricas(y_val, base.predict_proba(X_val)[:, 1]))

## 6. Modelos (≤ 20,000 parámetros cada uno)
MLP, Conv1D, GRU/LSTM. EarlyStopping sobre `val_loss`. Elegir con validación.

In [ ]:
# TODO: definir, entrenar y comparar en validación

## 7. Evaluación única en prueba y `resultados`

In [ ]:
# TODO: evaluar UNA vez cada modelo congelado en X_prueba
# resultados = pd.DataFrame(filas, columns=["modelo","tipo","parametros","sensibilidad","especificidad","auc","cumple","elegido"])
# resultados.to_csv("resultados_ecg.csv", index=False)

## 8. Justificación de la elección

## 9. Análisis de 3 errores

In [ ]:
# TODO: graficar FN/FP contra el promedio de su clase

## 10. Guardar el modelo elegido

In [ ]:
# modelo_elegido.save("modelo/modelo_ecg.keras")